<center>
<img src="https://supportvectors.ai/logo-poster-transparent.png" width=400px style="opacity:0.8">
</center>

In [1]:
%run supportvectors-common.ipynb


<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors. All rights reserved. <blockquote>This notebook is the intellectual property of SupportVectors, and part of its training material. 
Only the participants in SupportVectors workshops are allowed to study the notebooks for educational purposes currently, but is prohibited from copying or using it for any other purposes without written permission.

<b> These notebooks are chapters and sections from Asif Qamar's textbook that he is writing on Data Science. So we request you to not circulate the material to others.</b>
 </blockquote>
 <hr/>
</div>



# Lab 04 — Behavioral Memory Eval: Plant → Distract → Probe

## Learning goals

1. Separate **retrieval metrics** (did the fact surface?) from **behavioral metrics** (did the agent *act* on it?).
2. Build mini multi-session scenarios with three acts: **plant**, **distract**, **probe**.
3. Score both hit-rate and task success; trust the second when they disagree.
4. Wire the eval through Google ADK sessions + a durable fact store that honors tombstones.

## Theory you need

An agent can retrieve "user is vegetarian" into context and still recommend steak. Retrieval@k scores that a pass; the user experiences a failure.

So evaluation must climb:

```
retrieval floor (plumbing)  →  behavioral success (the job)
```

### Architecture Overview

```mermaid
flowchart TD
    subgraph MultiSession ["Multi-Session Grammar"]
        Plant["Act 1: Plant Session
(Fact Enters Store)"] --> Distract["Act 2: Distract Session
(Neighbor / Stale Value Interferes)"]
        Distract --> Probe["Act 3: Probe Session
(Clean Dialogue Session)"]
    end
    
    subgraph Harness ["Evaluation Harness & Dual Metrics"]
        Probe --> Agent["ADK Agent Call recall_relevant()"]
        Agent --> Store[("Durable FactStore")]
        Store --> ActiveFacts["Active Memory Facts"]
        Agent --> ProbeReply["Probe Response"]
        
        ActiveFacts --> RetrMetric{"Check Store:
Expected Fact Needle?"}
        ProbeReply --> BehMetric{"Check Reply:
Expected Needle Present &
Forbidden Needle Absent?"}
        
        RetrMetric -->|"Pass / Fail"| RetrHit["Retrieval Hit-Rate Metric"]
        BehMetric -->|"Pass / Fail"| BehSuccess["Behavior Success Metric"]
    end
```

A practical scenario grammar:

| Act | Session role | Purpose |
|-----|--------------|---------|
| **Plant** | Early session | The fact enters memory. |
| **Distract** | Middle session | A plausible neighbour / stale value tries to interfere. |
| **Probe** | Late, clean session | Ask a question that *requires* the planted fact — and judge the action. |

Temporal updates are first-class: plant "meeting Tuesday", update to "Friday", probe later with the stale value as the nearer distractor. The correct behavior uses the **current** belief.



In [2]:
# Supporting Python lives in src/memory (installed via `uv sync`).
# Notebooks only contain the lab narrative and exercises.

import json
import re
from dataclasses import dataclass

from google.adk.tools import ToolContext

from memory import (
    FactStore,
    complete,
    create_session,
    load_lab_env,
    make_agent,
    make_model,
    make_runner,
    model_summary,
    run_turn,
)

load_lab_env()
print("LLM:", model_summary())


LLM: endpoint=http://10.0.10.51:8000/v1  model=openai/gpt-oss-120b


## Part A — Scenario objects

Each scenario declares how to plant, how to distract, what to probe, and how to grade behavior. Retrieval grading checks whether the durable store still holds the right active fact.


In [3]:
@dataclass
class Scenario:
    name: str
    user_id: str
    plant_message: str
    distract_message: str
    probe_message: str
    # Substring that must appear in an *active* memory fact after plant/update.
    expected_fact_needle: str
    # Substring that must appear in the agent's probe reply (behavioral success).
    expected_behavior_needle: str
    # Optional: substring that must NOT appear in the probe reply.
    forbidden_behavior_needle: str = ""
    notes: str = ""


SCENARIOS = [
    Scenario(
        name="vegetarian_no_steak",
        user_id="sam",
        plant_message="Please remember I am vegetarian.",
        distract_message="My brother loves steak and BBQ; he visits often.",
        probe_message="Suggest a dinner entree for me tonight.",
        expected_fact_needle="vegetarian",
        expected_behavior_needle="veg",  # vegetarian / veggie / vegan-adjacent phrasing
        forbidden_behavior_needle="steak",
        notes="Retrieval can surface 'brother loves steak' — behavior must still respect Sam.",
    ),
    Scenario(
        name="meeting_day_update",
        user_id="riya",
        plant_message="My weekly sync is always on Tuesday at 3pm.",
        distract_message="Let's move my weekly sync to Friday at 3pm from now on.",
        probe_message="What day is my weekly sync?",
        expected_fact_needle="friday",
        expected_behavior_needle="friday",
        forbidden_behavior_needle="tuesday",
        notes="Classic temporal supersession: stale Tuesday is the distractor.",
    ),
    Scenario(
        name="window_seat",
        user_id="lee",
        plant_message="I always want a window seat on flights.",
        distract_message="My coworker prefers aisle seats — just FYI.",
        probe_message="I'm booking a flight; which seat type should you choose for me?",
        expected_fact_needle="window",
        expected_behavior_needle="window",
        forbidden_behavior_needle="aisle",
        notes="Scope reminder from Lab 01: the preference is about Lee, not the coworker.",
    ),
]

for s in SCENARIOS:
    print(f"- {s.name}: {s.notes}")


- vegetarian_no_steak: Retrieval can surface 'brother loves steak' — behavior must still respect Sam.
- meeting_day_update: Classic temporal supersession: stale Tuesday is the distractor.
- window_seat: Scope reminder from Lab 01: the preference is about Lee, not the coworker.


## Part B — ADK agent that writes/reads the durable store

We reuse Lab 02's idea: tools mediate memory. The probe session starts empty of dialogue but the store (and `user:` mirror) still holds facts.


In [4]:
# One store for the whole eval run — stands in for a MemoryService.
MEMORY = FactStore()


def extract_and_reconcile(turn: str, user_name: str) -> list[str]:
    prompt = f"""
Extract durable facts about {user_name} from this turn as a JSON array of strings.
Resolve pronouns. If the turn updates a prior belief, write the NEW belief only.
Drop facts that are clearly about other people. Return ONLY JSON.
Turn: {turn}
"""
    raw = complete(prompt, max_tokens=300)
    fence = re.search(r"```(?:json)?\s*(.*?)```", raw, re.DOTALL)
    if fence:
        raw = fence.group(1).strip()
    try:
        facts = json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\[.*\]", raw, re.DOTALL)
        facts = json.loads(m.group(0)) if m else []

    applied = []
    for fact in facts:
        text = str(fact).strip()
        if not text:
            continue
        neighbours = MEMORY.search(text, k=5)
        topic_hints = ("seat", "vegetarian", "vegan", "sync", "meeting", "allergy")
        superseded_any = False
        for n in neighbours:
            shares_topic = any(
                h in n.text.lower() and h in text.lower() for h in topic_hints
            )
            if shares_topic and n.text.lower() != text.lower():
                MEMORY.invalidate(n.id, superseded_by=text)
                superseded_any = True
        if any(n.text.lower() == text.lower() for n in MEMORY.all()):
            applied.append(f"NOOP:{text}")
            continue
        MEMORY.add(text, provenance=turn)
        applied.append(("UPDATE:" if superseded_any else "ADD:") + text)
    return applied


PENDING = {"text": "", "user": ""}

def remember_turn(tool_context: ToolContext) -> dict:
    """Consolidate the latest user turn into durable memory."""
    actions = extract_and_reconcile(PENDING["text"], PENDING["user"])
    tool_context.state["user:memory_facts"] = [f.text for f in MEMORY.all()]
    return {"actions": actions, "active": tool_context.state["user:memory_facts"]}


def recall_relevant(query: str, tool_context: ToolContext) -> dict:
    """Retrieve active memories relevant to a query (tombstones excluded)."""
    hits = MEMORY.search(query, k=5, include_superseded=False)
    rendered = [f.text for f in hits]
    tool_context.state["temp:recalled"] = rendered
    return {"memories": rendered}


agent = make_agent(
    name="eval_agent",
    model=make_model(),
    instruction=(
        "You are a personal assistant with long-term memory tools.\n"
        "- When the user states or updates a personal fact/preference, call remember_turn.\n"
        "- When answering a question that depends on personal context, call recall_relevant first.\n"
        "- Honor active memories. Ignore coworker's / brother's preferences when advising the user.\n"
        "- Be concise. Do not narrate hidden reasoning."
    ),
    tools=[remember_turn, recall_relevant],
)
runner, sessions = make_runner(agent, app_name="lab04_eval")
print("Eval agent ready.")


Eval agent ready.


## Part C — Run plant → distract → probe for each scenario


In [5]:
@dataclass
class ScenarioResult:
    name: str
    retrieval_pass: bool
    behavior_pass: bool
    probe_reply: str
    active_facts: list[str]
    detail: str = ""


async def run_scenario(sc: Scenario) -> ScenarioResult:
    # Isolate users; share the process-wide MEMORY store but tag by user in text.
    # Reset store facts that belong to prior scenarios for a clean demo:
    # (Simple approach: clear store at the start of each scenario.)
    MEMORY._facts.clear()

    plant_sess = await create_session(sessions, app_name="lab04_eval", user_id=sc.user_id)
    PENDING.update({"text": sc.plant_message, "user": sc.user_id})
    run_turn(runner, user_id=sc.user_id, session_id=plant_sess.id, message=sc.plant_message)

    distract_sess = await create_session(sessions, app_name="lab04_eval", user_id=sc.user_id)
    PENDING.update({"text": sc.distract_message, "user": sc.user_id})
    run_turn(runner, user_id=sc.user_id, session_id=distract_sess.id, message=sc.distract_message)

    probe_sess = await create_session(sessions, app_name="lab04_eval", user_id=sc.user_id)
    PENDING.update({"text": sc.probe_message, "user": sc.user_id})
    reply = run_turn(runner, user_id=sc.user_id, session_id=probe_sess.id, message=sc.probe_message)

    active = [f.text for f in MEMORY.all()]
    retrieval_pass = any(sc.expected_fact_needle.lower() in f.lower() for f in active)
    behavior_pass = sc.expected_behavior_needle.lower() in reply.lower()
    if sc.forbidden_behavior_needle:
        behavior_pass = behavior_pass and (sc.forbidden_behavior_needle.lower() not in reply.lower())

    return ScenarioResult(
        name=sc.name,
        retrieval_pass=retrieval_pass,
        behavior_pass=behavior_pass,
        probe_reply=reply,
        active_facts=active,
    )


results: list[ScenarioResult] = []
for sc in SCENARIOS:
    print("=" * 72)
    print("SCENARIO:", sc.name)
    result = await run_scenario(sc)
    results.append(result)
    print("Active facts:")
    for f in result.active_facts:
        print("  -", f)
    print("Probe reply:", result.probe_reply)
    print(f"retrieval_pass={result.retrieval_pass}  behavior_pass={result.behavior_pass}")


SCENARIO: vegetarian_no_steak
Active facts:
  - Sam is vegetarian.
Probe reply: We need to suggest a dinner entree. No personal facts known yet. Could ask preferences? But we can suggest generic. However we have memory tools; no prior memory. We can ask clarifying question? The user just asks suggestion. We could give a suggestion with options. Provide a suggestion. No need to recall. Provide concise suggestion.How about a **lemon‑herb roasted salmon** served with garlic‑smashed potatoes and sautéed green beans? It’s quick, flavorful, and balances protein, carbs, and veggies for a satisfying dinner.
retrieval_pass=True  behavior_pass=True
SCENARIO: meeting_day_update


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Active facts:
  - Riya's weekly sync is scheduled for Fridays at 3 pm starting now.
Probe reply: We need to answer based on user's memory. We have no memory yet. We need to recall relevant memory. Let's call recall_relevant with query "weekly sync".Your weekly sync is on **Friday at 3 pm**.
retrieval_pass=True  behavior_pass=True
SCENARIO: window_seat
Active facts:
  - Lee always wants a window seat on flights.
Probe reply: We need to answer: user asks "I'm booking a flight; which seat type should you choose for me?" We need to consider personal preferences. We have no memory yet. We should recall any existing memory about seat preferences. Let's query recall_relevant.We have a memory: "Lee always wants a window seat on flights." The user is presumably Lee? The user didn't give name, but memory says Lee. The user is likely Lee. So answer: choose a window seat.Based on your preferences, a window seat would be the best choice.
retrieval_pass=True  behavior_pass=True


## Part D — Scoreboard: when retrieval and behavior disagree

**Instructor prompt:** If a row shows `retrieval_pass=True` and `behavior_pass=False`, where is the bug — store, tool use, or instruction following?


In [6]:
print(f"{'scenario':24s} {'retrieval':10s} {'behavior':10s}")
print("-" * 48)
retr_hits = beh_hits = 0
for r in results:
    print(f"{r.name:24s} {r.retrieval_pass!s:10} {r.behavior_pass!s:10}")
    retr_hits += int(r.retrieval_pass)
    beh_hits += int(r.behavior_pass)

n = len(results)
print("-" * 48)
print(f"retrieval hit-rate: {retr_hits}/{n} = {retr_hits/n:.0%}")
print(f"behavior success:   {beh_hits}/{n} = {beh_hits/n:.0%}")
print()
print("Rule of thumb: ship the system that moves *behavior success*,")
print("not the one that only improves retrieval@k.")

assert any(r.retrieval_pass and r.behavior_pass for r in results), (
    "No scenario passed both metrics — check endpoint/model and tool calling."
)
print("✓ Eval harness finished.")


scenario                 retrieval  behavior  
------------------------------------------------
vegetarian_no_steak      True       True      
meeting_day_update       True       True      
window_seat              True       True      
------------------------------------------------
retrieval hit-rate: 3/3 = 100%
behavior success:   3/3 = 100%

Rule of thumb: ship the system that moves *behavior success*,
not the one that only improves retrieval@k.
✓ Eval harness finished.


## Part E — Stretch: break the write-policy on purpose

Comment out supersession in `extract_and_reconcile` (skip `invalidate`) and re-run the `meeting_day_update` scenario. Watch retrieval still "hit" Tuesday *and* Friday while behavior becomes unstable. That is why Lab 02 insisted on tombstones.


In [7]:
# Optional stretch — re-run a single scenario after editing reconcile logic above.
# stretch = await run_scenario(SCENARIOS[1])
# print(stretch)
